# 04. Continuous Torus Extension

This notebook runs the continuous-coordinate torus method and cohort-level p-value aggregation.


## Input expectation

Run Notebook 03 first, or load `st_adata_scored.h5ad` below so `st_by_sample` and `sample_groups` exist.


In [1]:
import os
from pathlib import Path

# Resolve project root robustly (works from notebooks/, project root, or Downloads/).
_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "Spatial HCC",
    Path.cwd().parent / "Spatial HCC",
]
for _root in _candidates:
    if (_root / "GSE238264_RAW").exists():
        os.chdir(_root)
        break

print("Working directory:", Path.cwd())


Working directory: /Users/prateek/Downloads/Spatial HCC


In [2]:
import scanpy as sc

if "st_adata" not in globals():
    st_adata = sc.read_h5ad("st_adata_scored.h5ad")

if "st_by_sample" not in globals():
    st_by_sample = {
        str(s): st_adata[st_adata.obs["sample"].astype(str) == str(s)].copy()
        for s in sorted(st_adata.obs["sample"].astype(str).unique())
    }

if "sample_groups" not in globals():
    sample_groups = {}
    if "response" in st_adata.obs.columns:
        tmp = st_adata.obs[["sample", "response"]].dropna().copy()
        tmp["sample"] = tmp["sample"].astype(str)
        tmp["response"] = tmp["response"].astype(str)
        first_resp = tmp.groupby("sample", as_index=True)["response"].first()
        for s, r in first_resp.items():
            rr = r.strip().upper()
            sample_groups[s] = "NR" if rr in ["NR", "NONRESPONDER", "NON-RESPONDER"] else ("R" if rr in ["R", "RESPONDER"] else r)
    for s in st_by_sample.keys():
        sample_groups.setdefault(s, "NR" if s.upper().endswith("NR") else ("R" if s.upper().endswith("R") else None))


RUN_MODE = os.environ.get("HCC_RUN_MODE", "full").strip().lower()
N_PERM_CONT = 2000 if RUN_MODE == "full" else 100

required_obs_cols = ["sample", "spot_tam", "spot_cytotoxic", "spot_exhaustion"]
missing_obs_cols = [c for c in required_obs_cols if c not in st_adata.obs.columns]
if missing_obs_cols:
    raise KeyError(f"st_adata is missing required columns: {missing_obs_cols}")


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [3]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.stats import spearmanr

def torus_shift_null_corr_continuous(
    ad,
    x_key,
    y_key,
    n_perm=2000,
    seed=1,
    k_nn=1,
):
    """
    Torus shift in continuous coordinate space (no NaN collapse).
    1) shift coordinates by random (dx,dy) with wrap-around (torus)
    2) assign shifted x values back onto original spots using NN mapping
    3) compute lag(x_shift) vs y using same W graph in ad.obsp['spatial_connectivities']

    Requires: ad.obsp['spatial_connectivities'] already computed (grid or delaunay etc).
    """
    if "spatial" not in ad.obsm:
        raise KeyError("Need ad.obsm['spatial']")

    if "spatial_connectivities" not in ad.obsp:
        raise KeyError("Need ad.obsp['spatial_connectivities'] already computed")

    W = ad.obsp["spatial_connectivities"]
    Wsum = np.maximum(W.sum(1).A1, 1e-9)

    xy = np.asarray(ad.obsm["spatial"], float)
    x = ad.obs[x_key].to_numpy()
    y = ad.obs[y_key].to_numpy()

    # observed
    x_lag = (W @ x) / Wsum
    obs = spearmanr(x_lag, y, nan_policy="omit")[0]

    # torus box
    xmin, ymin = xy.min(axis=0)
    xmax, ymax = xy.max(axis=0)
    Lx = float(max(xmax - xmin, 1e-9))
    Ly = float(max(ymax - ymin, 1e-9))

    tree = cKDTree(xy)
    rng = np.random.default_rng(seed)
    null = []

    for _ in range(n_perm):
        dx = rng.uniform(0, Lx)
        dy = rng.uniform(0, Ly)

        shifted = xy.copy()
        shifted[:, 0] = xmin + ((shifted[:, 0] - xmin + dx) % Lx)
        shifted[:, 1] = ymin + ((shifted[:, 1] - ymin + dy) % Ly)

        # map shifted positions back to original lattice by NN
        # query original points for shifted coords
        d, idx = tree.query(shifted, k=k_nn)
        if k_nn == 1:
            x_shift = x[idx]
        else:
            # average over k nearest
            x_shift = np.mean(x[idx], axis=1)

        xlag_p = (W @ x_shift) / Wsum
        rho = spearmanr(xlag_p, y, nan_policy="omit")[0]
        null.append(rho)

    null = np.asarray(null, float)
    p = (np.sum(np.abs(null) >= np.abs(obs)) + 1) / (len(null) + 1)
    return float(obs), float(p), null

In [4]:
torus_cont_rows = []
torus_cont_nulls = {}

for sample, ad in st_by_sample.items():
    # ensure spatial graph exists ONCE (use your current strategy; keep consistent)
    import squidpy as sq
    sq.gr.spatial_neighbors(ad, coord_type="grid")  # keep same as before
    # test 1: TAMlag vs EXH (x=TAM, y=EXH)
    obs1, p1, null1 = torus_shift_null_corr_continuous(
        ad, x_key="spot_tam", y_key="spot_exhaustion", n_perm=N_PERM_CONT, seed=1, k_nn=1
    )
    # test 2: CYTOlag vs EXH
    obs2, p2, null2 = torus_shift_null_corr_continuous(
        ad, x_key="spot_cytotoxic", y_key="spot_exhaustion", n_perm=N_PERM_CONT, seed=1, k_nn=1
    )

    torus_cont_rows.append({
        "sample": sample,
        "group": sample_groups.get(sample),
        "torus_cont_TAMlag_vs_EXH_rho": obs1,
        "torus_cont_TAMlag_vs_EXH_p": p1,
        "torus_cont_CYTOlag_vs_EXH_rho": obs2,
        "torus_cont_CYTOlag_vs_EXH_p": p2,
    })
    torus_cont_nulls[(sample, "TAMlag_vs_EXH")] = null1
    torus_cont_nulls[(sample, "CYTOlag_vs_EXH")] = null2

torus_cont_df = pd.DataFrame(torus_cont_rows).sort_values("sample").reset_index(drop=True)
display(torus_cont_df)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           
INFO     Creating graph using `grid` coordinates and `None` transform and `1` libraries.                           


,sample,group,torus_cont_TAMlag_vs_EXH_rho,torus_cont_TAMlag_vs_EXH_p,torus_cont_CYTOlag_vs_EXH_rho,torus_cont_CYTOlag_vs_EXH_p
0,HCC1R,R,-0.124861,0.009495,-0.030197,0.364318
1,HCC2R,R,-0.115302,0.016492,-0.044110,0.185407
2,HCC3R,R,-0.145252,0.011994,0.056219,0.183408
3,HCC4R,R,0.059061,0.006997,0.060354,0.008496
4,HCC5NR,NR,0.122185,0.017491,0.031427,0.454773
5,HCC6NR,NR,0.016974,0.437781,0.054720,0.011994
6,HCC7NR,NR,0.063514,0.006997,0.076721,0.003498


In [5]:
from scipy.stats import combine_pvalues
import numpy as np

def fisher_combine(df, pcol):
    ps = df[pcol].dropna().to_numpy()
    stat, p = combine_pvalues(ps, method="fisher")
    return len(ps), float(p)

n1, p1 = fisher_combine(torus_cont_df, "torus_cont_TAMlag_vs_EXH_p")
n2, p2 = fisher_combine(torus_cont_df, "torus_cont_CYTOlag_vs_EXH_p")
print("Fisher combined p (TAMlag vs EXH):", n1, p1)
print("Fisher combined p (CYTOlag vs EXH):", n2, p2)

sig_frac_TAM = np.mean(torus_cont_df["torus_cont_TAMlag_vs_EXH_p"] < 0.05)
sig_frac_CYT = np.mean(torus_cont_df["torus_cont_CYTOlag_vs_EXH_p"] < 0.05)
print("Significant fraction TAMlag:", sig_frac_TAM)
print("Significant fraction CYTOlag:", sig_frac_CYT)

Fisher combined p (TAMlag vs EXH): 7 5.908657182071598e-07
Fisher combined p (CYTOlag vs EXH): 7 0.0002504098117160054
Significant fraction TAMlag: 0.8571428571428571
Significant fraction CYTOlag: 0.42857142857142855


In [6]:
import matplotlib.pyplot as plt

# Torus rho plot: TAMlag->EXH
dfp = torus_cont_df.copy()
dfp = dfp.sort_values(["group","torus_cont_TAMlag_vs_EXH_rho"])

plt.figure(figsize=(7,4))
plt.axvline(0, linestyle="--")
plt.scatter(dfp["torus_cont_TAMlag_vs_EXH_rho"], range(len(dfp)))
for i, row in enumerate(dfp.itertuples()):
    plt.text(row.torus_cont_TAMlag_vs_EXH_rho, i, f" {row.sample} ({row.group}, p={row.torus_cont_TAMlag_vs_EXH_p:.3g})", va="center", fontsize=9)
plt.yticks([])
plt.xlabel("Torus-shift partial correlation: TAM-lag vs Exhaustion (rho)")
plt.tight_layout()
plt.savefig("fig_torus_TAMlag_vs_EXH_rho.png", dpi=300)
plt.close()

# Torus rho plot: CYTOlag->EXH
dfp = torus_cont_df.copy()
dfp = dfp.sort_values(["group","torus_cont_CYTOlag_vs_EXH_rho"])

plt.figure(figsize=(7,4))
plt.axvline(0, linestyle="--")
plt.scatter(dfp["torus_cont_CYTOlag_vs_EXH_rho"], range(len(dfp)))
for i, row in enumerate(dfp.itertuples()):
    plt.text(row.torus_cont_CYTOlag_vs_EXH_rho, i, f" {row.sample} ({row.group}, p={row.torus_cont_CYTOlag_vs_EXH_p:.3g})", va="center", fontsize=9)
plt.yticks([])
plt.xlabel("Torus-shift partial correlation: CYTO-lag vs Exhaustion (rho)")
plt.tight_layout()
plt.savefig("fig_torus_CYTOlag_vs_EXH_rho.png", dpi=300)
plt.close()

print("Saved: fig_torus_TAMlag_vs_EXH_rho.png, fig_torus_CYTOlag_vs_EXH_rho.png")

Saved: fig_torus_TAMlag_vs_EXH_rho.png, fig_torus_CYTOlag_vs_EXH_rho.png
